# Diabetes Pima

Uno de los dos datasets llamados «diabetes» (el otro, [Progresión de la diabetes](deep-dives/diabetes_progression.ipynb), es una profundización). Este es una clasificación, y su sorpresa es que los valores están en sus unidades, pero algunos de ellos no son valores en absoluto.

## Leer primero el dataset

OpenML, un repositorio abierto de datasets para aprendizaje automático, lo publica con el nombre `diabetes`; scikit-learn lo descarga la primera vez:

In [ ]:
from sklearn.datasets import fetch_openml

pima = fetch_openml(name="diabetes", version=1, as_frame=True)
pima.frame.head()

,preg,plas,pres,skin,insu,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,tested_positive
1,1,85,66,29,0,26.6,0.351,31,tested_negative
2,8,183,64,0,0,23.3,0.672,32,tested_positive
3,1,89,66,23,94,28.1,0.167,21,tested_negative
4,0,137,40,35,168,43.1,2.288,33,tested_positive


Los nombres de las columnas son abreviaturas, así que antes de nada hace falta la documentación. Su parte esencial:

In [ ]:
description = pima.DESCR
print(description[description.index("Past Usage"):description.index("Relevant Information")].strip())
print(description[description.index("Relevant Information"):description.index("Number of Instances")].strip())
print(description[description.index("For Each Attribute"):description.index("9. Class Distribution")].strip())

Past Usage:
     1. Smith,~J.~W., Everhart,~J.~E., Dickson,~W.~C., Knowler,~W.~C., &
        Johannes,~R.~S. (1988). Using the ADAP learning algorithm to forecast
        the onset of diabetes mellitus.  In {it Proceedings of the Symposium
        on Computer Applications and Medical Care} (pp. 261--265).  IEEE
        Computer Society Press.
 
        The diagnostic, binary-valued variable investigated is whether the
        patient shows signs of diabetes according to World Health Organization
        criteria (i.e., if the 2 hour post-load plasma glucose was at least 
        200 mg/dl at any survey  examination or if found during routine medical
        care).   The population lives near Phoenix, Arizona, USA.
 
        Results: Their ADAP algorithm makes a real-valued prediction between
        0 and 1.  This was transformed into a binary decision using a cutoff of 
        0.448.  Using 576 training instances, the sensitivity and specificity
        of their algorithm was 76% on 

Tres datos de ella deciden el flujo de trabajo:

- **Quién está en los datos:** 768 mujeres, de al menos 21 años, de ascendencia pima, un pueblo nativo americano de Arizona, estudiadas por el Instituto Nacional de Diabetes y Enfermedades Digestivas y Renales de Estados Unidos. Un modelo aprendido aquí describe a esta población; nada en los datos dice que valga para nadie más.
- **Qué es el objetivo:** si la mujer mostraba signos de diabetes según los criterios de la Organización Mundial de la Salud. Un diagnóstico, sí o no, hecho en cualquier examen del estudio (el estudio original lo usó para pronosticar la aparición de la diabetes): una clasificación, a diferencia de [Progresión de la diabetes](deep-dives/diabetes_progression.ipynb).
- **Qué son las variables:** número de embarazos (`preg`), glucosa en plasma dos horas después de una prueba de glucosa (`plas`), presión arterial diastólica (`pres`), grosor del pliegue cutáneo del tríceps (`skin`), insulina dos horas después de la prueba (`insu`), índice de masa corporal (`mass`), una puntuación de diabetes en la familia (`pedi`) y edad.

In [ ]:
X = pima.data
y = (pima.target == "tested_positive").astype(int).rename("diabetes")
y.value_counts()

diabetes
0    500
1    268
Name: count, dtype: int64

La documentación también dice que no hay valores ausentes. Los datos no están de acuerdo:

In [ ]:
X.describe().round(1)

,preg,plas,pres,skin,insu,mass,pedi,age
count,768.0,768.0,768.0,768.0,768.0,768.0,768.0,768.0
mean,3.8,120.9,69.1,20.5,79.8,32.0,0.5,33.2
std,3.4,32.0,19.4,16.0,115.2,7.9,0.3,11.8
min,0.0,0.0,0.0,0.0,0.0,0.0,0.1,21.0
25%,1.0,99.0,62.0,0.0,0.0,27.3,0.2,24.0
50%,3.0,117.0,72.0,23.0,30.5,32.0,0.4,29.0
75%,6.0,140.2,80.0,32.0,127.2,36.6,0.6,41.0
max,17.0,199.0,122.0,99.0,846.0,67.1,2.4,81.0


In [ ]:
measurements = ["plas", "pres", "skin", "insu", "mass"]
(X[measurements] == 0).sum()

plas      5
pres     35
skin    227
insu    374
mass     11
dtype: int64

La glucosa, la presión arterial, el pliegue cutáneo, la insulina y el índice de masa corporal tienen todos un mínimo de 0, y ninguna persona viva tiene ninguno de ellos a 0: son medidas que nunca se tomaron, guardadas como 0. La insulina falta en 374 mujeres, casi la mitad; el pliegue cutáneo en 227. Un modelo que recibiera estos ceros los trataría como valores reales muy bajos, y una insulina de 0 se convertiría en una prueba sobre la diabetes cuando lo único que dice es que nadie la midió. El «Missing Attribute Values: None» de la documentación solo es cierto en el sentido más estricto: no hay celdas vacías, porque los huecos se rellenaron con ceros.

## Valores ausentes, sin fugas

Convertir los ceros imposibles en valores ausentes es una regla fija: no aprende nada de los datos, así que se puede hacer una vez, antes de la división. *Rellenarlos* es distinto: el relleno habitual, la mediana de cada columna, se aprende de los datos, así que va dentro del pipeline, ajustado solo con las filas de entrenamiento ([el conjunto de prueba representa la producción](../ml-fundamentals/overfitting_and_the_split.ipynb#el-conjunto-de-prueba-representa-la-producción)). Rellenarlos con la mediana de las 768 mujeres antes de dividir sería una fuga pequeña, pero justo del tipo que la división existe para evitar. Por qué faltan los valores, y las demás formas de rellenarlos: [Valores ausentes](../../data-preparation/missing_values.ipynb).

`SimpleImputer(add_indicator=True)` añade además, para cada columna que tenía valores ausentes, una columna 0/1 que dice si el valor faltaba. Que se midiera o no la insulina puede decir algo de una paciente, y así el modelo puede usarlo.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = X.assign(**{column: X[column].replace(0, np.nan) for column in measurements})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.isna().sum()

preg      0
plas      4
pres     23
skin    175
insu    290
mass      9
pedi      0
age       0
dtype: int64

## Comparar modelos

Una referencia que siempre responde «sin diabetes», una regresión logística y un bosque aleatorio, comparados por validación cruzada de 5 pliegues sobre las filas de entrenamiento. Las medidas son las de [Evaluar un clasificador](../evaluation/classification_metrics.ipynb): exactitud, exhaustividad de la clase positiva y el área bajo la curva ROC:

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = {
    "always 'no diabetes'": DummyClassifier(),
    "logistic regression": make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), LogisticRegression()),
    "random forest": make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                   RandomForestClassifier(min_samples_leaf=5, random_state=42, n_jobs=-1)),
}
cross_validated = pd.DataFrame({
    name: pd.DataFrame(cross_validate(model, X_train, y_train, cv=5, scoring=["accuracy", "recall", "roc_auc"])).mean()
    for name, model in candidates.items()
}).T[["test_accuracy", "test_recall", "test_roc_auc"]]
cross_validated.round(3)

,test_accuracy,test_recall,test_roc_auc
always 'no diabetes',0.651,0.000,0.500
logistic regression,0.783,0.584,0.841
random forest,0.767,0.561,0.830


`cross_validate` llama «test» a cada pliegue apartado; es el pliegue de validación de cada ronda, no el conjunto de prueba, que sigue sin tocar. Responder siempre «sin diabetes» acierta el 65 % de las veces y no encuentra ningún caso. La regresión logística acierta alrededor del 78 % de las veces, encuentra alrededor del 58 % de los casos y tiene un AUC de 0,84; el bosque queda ligeramente por detrás en las tres, así que se elige la regresión logística.

## Elegir un umbral

En un cribado, un caso no detectado cuesta más que una segunda prueba, así que el umbral por defecto de 0,5 no es necesariamente el correcto ([el umbral](../evaluation/classification_metrics.ipynb#el-umbral)). Como cualquier ajuste, se elige con las filas de entrenamiento: las probabilidades de validación cruzada, cada una predicha por un modelo que no vio esa fila, muestran el compromiso sin tocar el conjunto de prueba:

In [ ]:
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import cross_val_predict

chosen_name = cross_validated["test_roc_auc"].idxmax()
final_model = candidates[chosen_name]
training_probabilities = cross_val_predict(final_model, X_train, y_train, cv=5, method="predict_proba")[:, 1]
pd.DataFrame({
    threshold: {
        "precision": precision_score(y_train, training_probabilities >= threshold),
        "recall": recall_score(y_train, training_probabilities >= threshold),
    }
    for threshold in (0.2, 0.3, 0.4, 0.5)
}).T.rename_axis("threshold").round(3)

,precision,recall
threshold,,
0.2,0.528,0.874
0.3,0.594,0.766
0.4,0.670,0.654
0.5,0.740,0.584


Con 0,5 el modelo encuentra el 58 % de los casos, y cuando marca uno acierta el 74 % de las veces. Con 0,3 encuentra alrededor del 77 %, con una precisión de alrededor del 59 %: en un cribado, donde un positivo lleva a una prueba de confirmación y no a un diagnóstico, ese es el mejor intercambio. El estudio original informó en 1988 de una sensibilidad y una especificidad del 76 % cada una (citadas en la documentación de arriba): con 0,3 este modelo encuentra más o menos la misma proporción de casos, así que está en el mismo rango, no es mejor. Aquí se elige el umbral de 0,3, con las filas de entrenamiento. El `predict` de un modelo sigue usando 0,5; para distribuir el umbral con el modelo, scikit-learn envuelve ambos en un objeto, `FixedThresholdClassifier(model, threshold=0.3)`, y `TunedThresholdClassifierCV` elige el umbral por validación cruzada.

## Probar una sola vez

El modelo, entrenado con todas las filas de entrenamiento, y el umbral elegido, medidos una sola vez con el conjunto de prueba:

In [ ]:
from sklearn.metrics import classification_report

final_model.fit(X_train, y_train)
screening_threshold = 0.3
test_flagged = (final_model.predict_proba(X_test)[:, 1] >= screening_threshold).astype(int)
print(classification_report(y_test, test_flagged, target_names=["no diabetes", "diabetes"]))

              precision    recall  f1-score   support

 no diabetes       0.86      0.71      0.78       100
    diabetes       0.59      0.78      0.67        54

    accuracy                           0.73       154
   macro avg       0.72      0.74      0.72       154
weighted avg       0.76      0.73      0.74       154



Con las 154 mujeres de prueba, el modelo con un umbral de 0,3 encuentra el 78 % de los casos de diabetes, y el 59 % de las mujeres que marca la tienen: lo que predijo la validación cruzada. Dos de cada cinco mujeres marcadas pasarían a una prueba de confirmación que resulta que no necesitan, que es el precio, elegido deliberadamente, de no detectar menos de uno de cada cuatro casos.

## Dos datasets llamados «diabetes» *(profundización)*

| | [Progresión de la diabetes](deep-dives/diabetes_progression.ipynb) | Diabetes Pima |
| --- | --- | --- |
| Quién | 442 pacientes que ya tienen diabetes | 768 mujeres de ascendencia pima, de 21 años o más |
| Objetivo | una medida de la progresión un año después: regresión | signos de diabetes, sí o no: clasificación |
| Unidades de las variables | eliminadas: centradas y escaladas, usando todas las filas | conservadas, pero los ceros representan valores ausentes |
| Lo que la documentación oculta o dice mal | qué sexo es cada código | dice que no hay valores ausentes |
| Lo que haría un pipeline escrito sin leer | trabajar con variables escaladas incluyendo las filas de prueba, y leer pesos en unidades que ya no existen | aprender que un nivel de insulina de 0 es real |